# MFCC feature extraction (ALI_new/All)

This notebook extracts MFCC features using the same MFCC settings as the ICASSP2020 CNN-LSTM baseline:

- Audio: mono, resampled to 16 kHz
- MFCC: 36 dims
- Frame length: 20 ms, frame shift: 10 ms (win=320, hop=160 at 16 kHz)

For compatibility with the few-shot meta-learning pipeline (which expects fixed-size vectors per utterance),
we summarize each utterance by concatenating **mean + std** over MFCC frames, yielding a 72-D feature vector.

Outputs are saved to:

- `Results/MFCC_ALI_new_All/als_mfcc_dataset.npz` (subject-wise, variable-length)
- `Results/MFCC_ALI_new_All/utterance_level_dataset.npz` (utterance-wise)
- `Results/MFCC_ALI_new_All/subject_metadata.csv`
- `Results/MFCC_ALI_new_All/utterance_metadata.csv`
- `Results/MFCC_ALI_new_All/extraction_statistics.txt`
- `Results/MFCC_ALI_new_All/als_dataset_complete.pkl`


In [1]:
import os
from pathlib import Path
from collections import Counter
import pickle
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

# =============================
# PATHS
# =============================
DATA_ROOT = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/ALI_new/All")
OUT_DIR = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/MFCC_ALI_new_All")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================
# LABELS (keep consistent with your main experiments)
# Control -> 0, ALSwDysarthria -> 1, ALSwoDysarthria -> 2
# =============================
GROUP_LABEL_MAP = {
    "Control": 0,
    "ALSwDysarthria": 1,
    "ALSwoDysarthria": 2,
}
GROUP_NAMES = {0: "Control", 1: "Symptomatic_ALS", 2: "Presymptomatic_ALS"}

# =============================
# MFCC settings (paper-aligned)
# =============================
SR = 16000
N_MFCC = 36
FRAME_LEN_MS = 20
FRAME_SHIFT_MS = 10
WIN_LENGTH = int(SR * FRAME_LEN_MS / 1000.0)   # 320
HOP_LENGTH = int(SR * FRAME_SHIFT_MS / 1000.0) # 160
N_FFT = 512
N_MELS = 40

# We pool MFCC frames into a fixed vector per utterance: mean + std => 72 dims
FEATURE_DIM = 2 * N_MFCC

print("DATA_ROOT:", DATA_ROOT, "| exists:", DATA_ROOT.exists())
print("OUT_DIR  :", OUT_DIR)
print("MFCC settings: SR=", SR, "N_MFCC=", N_MFCC, "win=", WIN_LENGTH, "hop=", HOP_LENGTH)
print("Utterance feature dim:", FEATURE_DIM)

# Torch/torchaudio are required for this exact MFCC configuration
try:
    import torch
    import torchaudio
except Exception as e:
    raise ImportError(
        "This notebook requires torch + torchaudio. Run it in your ali_torch (or equivalent) environment."
    ) from e

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SR,
    n_mfcc=N_MFCC,
    melkwargs={
        "n_fft": N_FFT,
        "win_length": WIN_LENGTH,
        "hop_length": HOP_LENGTH,
        "n_mels": N_MELS,
        "window_fn": torch.hann_window,
        "power": 2.0,
    },
).to(device)


DATA_ROOT: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/ALI_new/All | exists: True
OUT_DIR  : /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/MFCC_ALI_new_All
MFCC settings: SR= 16000 N_MFCC= 36 win= 320 hop= 160
Utterance feature dim: 72
device: cuda


In [2]:
def _load_wav_mono_16k(path: Path) -> torch.Tensor:
    """Load audio, convert to mono, resample to 16 kHz. Returns (1, n_samples) float32."""
    try:
        import soundfile as sf
        x, sr = sf.read(str(path), always_2d=True)
        x = x.mean(axis=1)
        wav = torch.from_numpy(x.astype(np.float32))[None, :]
    except Exception:
        wav, sr = torchaudio.load(str(path))
        if wav.ndim == 2 and wav.size(0) > 1:
            wav = wav.mean(dim=0, keepdim=True)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    return wav


@torch.no_grad()
def extract_mfcc_utterance_vector(path: Path) -> np.ndarray | None:
    """Extract MFCC frames (T, 36) then pool into (72,) = [mean, std]."""
    try:
        wav = _load_wav_mono_16k(path).to(device)
        mfcc = mfcc_transform(wav)              # (1, 36, T)
        mfcc = mfcc.squeeze(0).transpose(0, 1) # (T, 36)
        if mfcc.numel() == 0:
            return np.zeros(FEATURE_DIM, dtype=np.float32)
        mean = mfcc.mean(dim=0)
        std = mfcc.std(dim=0, unbiased=False)
        vec = torch.cat([mean, std], dim=0).detach().cpu().numpy().astype(np.float32)
        if vec.shape[0] != FEATURE_DIM:
            return None
        return vec
    except Exception:
        return None


In [3]:
def extract_all_subjects():
    """Extract MFCC vectors for ALL subjects and ALL utterances (variable-length)."""
    print("=" * 80)
    print("STEP: EXTRACTING MFCC FEATURES FOR ALL SUBJECTS")
    print("=" * 80)

    subject_features = []
    subject_valid_flags = []
    subject_labels = []
    subject_metadata = []

    stats = {
        "total_subjects": 0,
        "total_utterances": 0,
        "successful_subjects": 0,
        "failed_subjects": [],
        "group_counts": {0: 0, 1: 0, 2: 0},
    }

    for group_folder in sorted(os.listdir(DATA_ROOT)):
        group_path = DATA_ROOT / group_folder
        if not group_path.is_dir() or group_folder not in GROUP_LABEL_MAP:
            continue

        y = int(GROUP_LABEL_MAP[group_folder])
        print(f"\nProcessing {group_folder} (label={y} - {GROUP_NAMES[y]})")
        subject_folders = sorted([p for p in group_path.iterdir() if p.is_dir()])
        print(f"Found {len(subject_folders)} subjects")

        for subj_dir in tqdm(subject_folders, desc=group_folder):
            wavs = sorted(subj_dir.glob("*.wav"))
            subject_id = f"{group_folder}/{subj_dir.name}"

            if len(wavs) == 0:
                stats["failed_subjects"].append(subject_id)
                continue

            base_name = subj_dir.name.replace("_denoised", "")
            sex = "Female" if base_name.startswith("F") else "Male"
            all_vecs = []
            valid_flags = []
            failed_files_sample = []

            for wp in wavs:
                vec = extract_mfcc_utterance_vector(wp)
                if vec is not None and vec.shape[0] == FEATURE_DIM:
                    all_vecs.append(vec)
                    valid_flags.append(1)
                else:
                    all_vecs.append(np.zeros(FEATURE_DIM, dtype=np.float32))
                    valid_flags.append(0)
                    if len(failed_files_sample) < 5:
                        failed_files_sample.append(wp.name)

            features_array = np.asarray(all_vecs, dtype=np.float32)              # (n_utts, 72)
            features_reshaped = np.expand_dims(features_array, axis=0)           # (1, n_utts, 72)
            valid_flags_reshaped = np.expand_dims(np.asarray(valid_flags, dtype=np.int8), axis=0)

            subject_features.append(features_reshaped)
            subject_valid_flags.append(valid_flags_reshaped)
            subject_labels.append(y)

            failed_utterances = int(len(wavs) - sum(valid_flags))
            subject_metadata.append({
                "subject_id": subject_id,
                "subject_folder": subj_dir.name,
                "group": group_folder,
                "group_label": y,
                "group_name": GROUP_NAMES[y],
                "sex": sex,
                "num_utterances": int(features_array.shape[0]),
                "num_utterances_valid": int(sum(valid_flags)),
                "num_utterances_failed": failed_utterances,
                "feature_dim": int(features_array.shape[1]),
                "original_files": int(len(wavs)),
                "failed_files_sample": failed_files_sample,
            })

            stats["total_subjects"] += 1
            stats["total_utterances"] += int(len(wavs))
            stats["successful_subjects"] += 1
            stats["group_counts"][y] += 1

    if len(subject_features) == 0:
        raise RuntimeError("No subjects processed.")

    dataset = {
        "subject_features": subject_features,
        "subject_valid_flags": subject_valid_flags,
        "subject_labels": np.asarray(subject_labels, dtype=np.int32),
        "subject_metadata": subject_metadata,
        "feature_dim": FEATURE_DIM,
        "group_names": GROUP_NAMES,
        "group_label_map": GROUP_LABEL_MAP,
    }

    print("\n" + "=" * 80)
    print("DATASET SUMMARY")
    print("=" * 80)
    print("Total subjects processed:", stats["successful_subjects"])
    print("Total utterances:", stats["total_utterances"])
    print("Average utterances per subject:", stats["total_utterances"] / stats["successful_subjects"])
    print("\nGroup distribution:")
    for lab in [0, 1, 2]:
        c = stats["group_counts"][lab]
        pct = 100.0 * c / stats["successful_subjects"]
        print(f"  {GROUP_NAMES[lab]}: {c} subjects ({pct:.1f}%)")
    print("\nSex distribution:")
    sex_counts = Counter(m["sex"] for m in subject_metadata)
    for sx, c in sex_counts.items():
        pct = 100.0 * c / stats["successful_subjects"]
        print(f"  {sx}: {c} subjects ({pct:.1f}%)")

    return dataset, stats


In [4]:
def save_dataset(dataset, stats):
    print("\n" + "=" * 80)
    print("SAVING DATASET")
    print("=" * 80)

    n_subjects = len(dataset["subject_features"])
    total_utts = int(sum(sf.shape[1] for sf in dataset["subject_features"]))
    print(f"Subjects to save: {n_subjects} | Total utterances: {total_utts}")

    # ---- subject-wise (variable-length) ----
    dataset_file = OUT_DIR / "als_mfcc_dataset.npz"
    subj_feats = dataset["subject_features"]
    subject_features_obj = np.empty(len(subj_feats), dtype=object)
    subject_features_obj[:] = subj_feats

    subj_valid = dataset.get("subject_valid_flags", [])
    subject_valid_flags_obj = np.empty(len(subj_valid), dtype=object)
    if len(subj_valid) > 0:
        subject_valid_flags_obj[:] = subj_valid

    np.savez_compressed(
        dataset_file,
        subject_features=subject_features_obj,
        subject_valid_flags=subject_valid_flags_obj,
        subject_labels=dataset["subject_labels"],
        feature_dim=dataset["feature_dim"],
        group_names=list(dataset["group_names"].values()),
        group_label_map=dataset["group_label_map"],
    )
    print("✅ Subject-wise dataset saved:", dataset_file)

    # ---- subject metadata ----
    subject_meta_csv = OUT_DIR / "subject_metadata.csv"
    pd.DataFrame(dataset["subject_metadata"]).to_csv(subject_meta_csv, index=False)
    print("✅ Subject metadata saved:", subject_meta_csv)

    # ---- stats text ----
    stats_file = OUT_DIR / "extraction_statistics.txt"
    with open(stats_file, "w") as f:
        f.write("MFCC FEATURE EXTRACTION STATISTICS\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Subjects processed: {stats['successful_subjects']}\n")
        f.write(f"Total utterances: {stats['total_utterances']}\n")
        f.write(f"Avg utterances/subject: {stats['total_utterances']/max(stats['successful_subjects'],1):.2f}\n\n")
        f.write("GROUP COUNTS:\n")
        for lab in [0, 1, 2]:
            f.write(f"  {GROUP_NAMES[lab]}: {stats['group_counts'][lab]}\n")
    print("✅ Statistics saved:", stats_file)

    # ---- pickle ----
    pickle_file = OUT_DIR / "als_dataset_complete.pkl"
    with open(pickle_file, "wb") as f:
        pickle.dump(dataset, f)
    print("✅ Pickle saved:", pickle_file)

    return dataset_file, subject_meta_csv, stats_file, pickle_file


def create_utterance_level_dataset(dataset):
    print("\n" + "=" * 80)
    print("CREATING UTTERANCE-LEVEL DATASET")
    print("=" * 80)

    utterance_features = []
    utterance_labels = []
    utterance_metadata = []

    valid_flags_list = dataset.get("subject_valid_flags", None)

    for i, (sf, y) in enumerate(zip(dataset["subject_features"], dataset["subject_labels"])):
        X_u = sf[0]  # (n_utts, 72)
        meta = dataset["subject_metadata"][i]
        for j in range(X_u.shape[0]):
            utterance_features.append(X_u[j])
            utterance_labels.append(int(y))
            is_valid = int(valid_flags_list[i][0][j]) if valid_flags_list is not None else 1
            utterance_metadata.append({
                "subject_id": meta["subject_id"],
                "group": meta["group"],
                "group_label": int(y),
                "sex": meta["sex"],
                "utterance_index": int(j),
                "is_valid": is_valid,
                "total_utterances": int(meta["num_utterances"]),
            })

    utterance_features = np.asarray(utterance_features, dtype=np.float32)
    utterance_labels = np.asarray(utterance_labels, dtype=np.int32)

    print("Utterances:", len(utterance_features), "| features:", utterance_features.shape)

    utt_npz = OUT_DIR / "utterance_level_dataset.npz"
    np.savez_compressed(
        utt_npz,
        features=utterance_features,
        labels=utterance_labels,
        feature_dim=int(dataset["feature_dim"]),
        group_names=list(dataset["group_names"].values()),
    )
    print("✅ Utterance dataset saved:", utt_npz)

    utt_meta_csv = OUT_DIR / "utterance_metadata.csv"
    pd.DataFrame(utterance_metadata).to_csv(utt_meta_csv, index=False)
    print("✅ Utterance metadata saved:", utt_meta_csv)

    return utt_npz, utt_meta_csv


In [5]:
# =============================
# RUN
# =============================
dataset, stats = extract_all_subjects()
dataset_file, subject_meta_csv, stats_file, pickle_file = save_dataset(dataset, stats)
utt_npz, utt_meta_csv = create_utterance_level_dataset(dataset)

print("\nDONE")
print("- subject-wise:", dataset_file)
print("- subject meta:", subject_meta_csv)
print("- utterance-wise:", utt_npz)
print("- utterance meta:", utt_meta_csv)


STEP: EXTRACTING MFCC FEATURES FOR ALL SUBJECTS

Processing ALSwDysarthria (label=1 - Symptomatic_ALS)
Found 27 subjects


ALSwDysarthria: 100%|██████████| 27/27 [00:41<00:00,  1.55s/it]



Processing ALSwoDysarthria (label=2 - Presymptomatic_ALS)
Found 16 subjects


ALSwoDysarthria: 100%|██████████| 16/16 [00:22<00:00,  1.43s/it]



Processing Control (label=0 - Control)
Found 20 subjects


Control: 100%|██████████| 20/20 [00:28<00:00,  1.42s/it]



DATASET SUMMARY
Total subjects processed: 63
Total utterances: 6926
Average utterances per subject: 109.93650793650794

Group distribution:
  Control: 20 subjects (31.7%)
  Symptomatic_ALS: 27 subjects (42.9%)
  Presymptomatic_ALS: 16 subjects (25.4%)

Sex distribution:
  Female: 27 subjects (42.9%)
  Male: 36 subjects (57.1%)

SAVING DATASET
Subjects to save: 63 | Total utterances: 6926
✅ Subject-wise dataset saved: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/MFCC_ALI_new_All/als_mfcc_dataset.npz
✅ Subject metadata saved: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/MFCC_ALI_new_All/subject_metadata.csv
✅ Statistics saved: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/MFCC_ALI_new_All/extraction_statistics.txt
✅ Pickle saved: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/MFCC_ALI_new_All/als_dataset_complete.pkl

CREATING UTTERANCE-LEVEL DATASET
Utterances: 6926 | features: (6926, 72)
✅ Utterance dataset saved: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results